# 02 — Pré-processamento

**Dimensão 4 da rúbrica — 15 pontos.**

Cada decisão precisa de justificativa escrita. Decidir *não* criar features é
aceitável, desde que o motivo esteja explícito.

##  Definição e Justificativa da Variável Alvo (Target)

Para treinar nosso modelo de classificação, precisamos estabelecer um critério claro do que define um "mau pagador" (*default*). Como nossa base de histórico financeiro (`credit_record.csv`) é longitudinal (mensal), construímos a variável alvo baseada nas seguintes premissas de negócio e gestão de risco:

* **O Limiar de Inadimplência:** A coluna `STATUS` classifica os pagamentos mensais. Optamos por definir como inadimplência grave (Target = 1) os status `2`, `3`, `4` e `5`, que representam faturas atrasadas há 60 dias ou mais. 
* **Justificativa de Negócio:** No setor bancário, atrasos curtos (menos de 60 dias) muitas vezes decorrem de esquecimentos ou desencontros de fluxo de caixa pontuais. Ultrapassar a barreira dos 60 dias indica uma degradação real na capacidade de pagamento, alinhando-se a métricas regulatórias de provisionamento de risco que exigem maior capitalização das instituições para absorver possíveis perdas.
* **A Lógica da Janela de Performance:** Ao agrupar o histórico por cliente (`ID`), aplicamos a função `.max()` na nossa variável temporária. Isso significa que adotamos uma postura conservadora: se o cliente atingiu o patamar de inadimplência grave em *qualquer momento* do seu histórico, ele é rotulado como `1` para o treinamento do modelo. Clientes que mantiveram status `0`, `1`, `C` (pago) ou `X` (sem empréstimo no mês) em todo o período recebem `0`.

In [7]:
from pathlib import Path
import numpy as np
import pandas as pd

# 1. Definir os caminhos e carregar os datasets brutos com pandas
path_app = Path("../data/raw/application_record.csv")
path_cred = Path("../data/raw/credit_record.csv")

df_app = pd.read_csv(path_app)
df_cred = pd.read_csv(path_cred)

print(f'Dimensão original do Cadastro (application_record.csv): {df_app.shape}')
print(f'Dimensão original do Histórico (credit_record.csv): {df_cred.shape}')

# 2. Construção da Variável Alvo (Target) baseada na regra de inadimplência (60+ dias de atraso)
maus_pagadores_status = ['2', '3', '4', '5']
df_cred['TARGET_TEMP'] = df_cred['STATUS'].astype(str).apply(
    lambda x: 1 if x in maus_pagadores_status else 0
)

# Agrupar por ID para capturar o pior status histórico do cliente (.max())
clientes_alvo = df_cred.groupby('ID')['TARGET_TEMP'].max().reset_index()
clientes_alvo.rename(columns={'TARGET_TEMP': 'TARGET'}, inplace=True)

# 3. Realizar o Inner Join para isolar a população de modelagem com histórico válido
df_modelagem = pd.merge(df_app, clientes_alvo, on='ID', how='inner')

print(f'\nDimensão da base unificada de modelagem (Inner Join): {df_modelagem.shape}')

# Cálculo e formatação da proporção da variável Target (2 casas decimais e símbolo %)
proporcoes = (df_modelagem['TARGET'].value_counts(normalize=True) * 100).round(2).astype(str) + '%'
print(f"Proporção da Target na população de modelagem:\n{proporcoes.to_string()}")

Dimensão original do Cadastro (application_record.csv): (438557, 18)
Dimensão original do Histórico (credit_record.csv): (1048575, 3)

Dimensão da base unificada de modelagem (Inner Join): (36457, 19)
Proporção da Target na população de modelagem:
TARGET
0    98.31%
1     1.69%


### Unificação das Bases (Inner Join) e Criação do Dataset de Modelagem

Para garantir a consistência dos dados, precisamos cruzar as informações demográficas do cadastro (`application_record.csv`) com a variável alvo financeira (`credit_record.csv`). 

A abordagem escolhida para essa consolidação foi o **Inner Join** (interseção) utilizando a coluna `ID` como chave de ligação. A justificativa para essa escolha é:

* **Validação Cruzada:** Ao utilizar o Inner Join, o código retém estritamente os clientes que aparecem **em ambas as tabelas simultaneamente**. 
* **Descarte de Registros Órfãos:** Clientes que possuem o histórico financeiro (Target), mas não o perfil cadastral (Features) — ou vice-versa — são sumariamente descartados. Para o treinamento de modelos de *Machine Learning*, registros sem o par completo de dados (perfil + alvo) não geram aprendizado e exigiriam imputações artificiais de dados nulos, comprometendo a precisão das regras capturadas.
* **Consolidação:** O resultado dessa interseção é armazenado no novo dataframe `df_modelagem`. Este *dataset* unificado representa a nossa população oficial e higienizada, sobre a qual todas as análises subsequentes, tratamentos de *outliers* e modelagens serão executadas.

A verificação final das proporções da variável alvo na base `df_modelagem` demonstra que, apesar do corte, a taxa original de representação da inadimplência manteve-se estável, indicando que o descarte de registros não causou viés na amostragem.

Um ponto importante: No gráfico de Balanceamento das Classes, gerado no primeiro notebook `01_eda.ipynb`, a quantidade de bons e maus pagadores classificados usa como base todo o dataset `credit_record.csv`. Na análise executada acima, os valores percentuais de **98,31%** (Target 0) e **1,68%** (Target 1) refletem a proporção estrita dentro do novo dataset `df_modelagem`. Isso significa que, dos 36.457 clientes resultantes da interseção, a maioria pertence à classe positiva de bons pagadores, evidenciando o forte desbalanceamento natural da base que precisará ser tratado nas etapas de modelagem.

## 1. Dados faltantes

In [9]:
# 1. Identificação de Dados Faltantes no dataset unificado
nulos = df_modelagem.isnull().sum()
percentual = (df_modelagem.isnull().mean() * 100).round(2)

df_nulos = pd.DataFrame({
    'Quantidade_Nulos': nulos, 
    'Percentual (%)': percentual
})

# Filtrar apenas as colunas que possuem algum valor vazio
colunas_com_nulos = df_nulos[df_nulos['Quantidade_Nulos'] > 0].sort_values(by='Percentual (%)', ascending=False)
print("Resumo de Dados Faltantes:\n")
print(colunas_com_nulos)

Resumo de Dados Faltantes:

                 Quantidade_Nulos  Percentual (%)
OCCUPATION_TYPE             11323           31.06


### 1.1. Diagnóstico e Decisão de Tratamento

A execução do bloco de código revelou que a variável `OCCUPATION_TYPE` (tipo de profissão) é a única que apresenta dados faltantes no nosso dataset unificado. Em formulários de concessão de crédito, a omissão desse dado costuma ocorrer quando o cliente atua na informalidade, não encontra sua área de atuação na lista de opções do sistema ou simplesmente prefere não declarar.

Para não perdermos esses clientes valiosos — que já possuem histórico financeiro mapeado na nossa variável alvo —, a decisão adotada foi preencher essas lacunas com a categoria `"Outros"`. Essa estratégia de imputação converte a ausência de dado em uma informação categórica explícita, permitindo que o algoritmo de *Machine Learning* interprete esse padrão de "não preenchimento" durante o treinamento, sem distorcer o peso das demais profissões.

In [10]:
# Substituir os valores nulos (NaN) da coluna OCCUPATION_TYPE pela string 'Outros'
df_modelagem['OCCUPATION_TYPE'] = df_modelagem['OCCUPATION_TYPE'].fillna('Outros')

# Validação do tratamento: somar todos os nulos do dataframe
total_nulos = df_modelagem.isnull().sum().sum()

print(f"Total de valores nulos na base após o tratamento: {total_nulos}")

Total de valores nulos na base após o tratamento: 0


Com a validação retornando zero, confirmamos que a base `df_modelagem` está totalmente higienizada em relação a dados faltantes. 

A estrutura agora está íntegra e pronta para a próxima fase do pré-processamento: a identificação e a padronização de *outliers* (valores extremos e atípicos), com foco especial na anomalia já detectada na coluna de tempo de emprego (`DAYS_EMPLOYED`).

### 2. Tratamento de Outliers (`DAYS_EMPLOYED`)

Durante a Análise Exploratória (Notebook 01), identificamos um comportamento anômalo na coluna `DAYS_EMPLOYED` (Tempo de Emprego): a presença do valor exato `365243`. 

Neste *dataset*, as datas relativas são representadas por números inteiros negativos (contagem de dias no passado até o momento da solicitação). O valor `365243` (que equivale matematicamente a exatos 1000 anos no futuro) é um clássico *placeholder* (marcador de sistema padrão) utilizado pela instituição financeira para registrar clientes que não possuem emprego formal ativo no momento da aplicação — grupo que engloba aposentados, estudantes e desempregados.
Essa afirmação, sobre o valor `365243`, foi verificada no seguinte link: https://www.kaggle.com/code/bolouki/home-credit-default-risk-eda-and-model. Onde a análise desse valor indica, por exemplo a situação de desempregado.

Se mantivermos esse valor na base, ele agirá como um *outlier* extremo e destruirá a escala da variável, prejudicando o desempenho de algoritmos de *Machine Learning* baseados em distância ou gradiente. A solução analítica mais coerente é substituir `365243` por `0`, representando corretamente que o cliente acumula "zero dias" de emprego formal atual.

In [12]:
# Verificar a quantidade de registros com o outlier antes do tratamento
qtd_outliers = (df_modelagem['DAYS_EMPLOYED'] == 365243).sum()
print(f"Clientes identificados com o marcador de sistema (365243): {qtd_outliers}")

# Substituir o outlier por 0
df_modelagem['DAYS_EMPLOYED'] = df_modelagem['DAYS_EMPLOYED'].replace(365243, 0)

# Validar o tratamento verificando o novo valor máximo da coluna
novo_max = df_modelagem['DAYS_EMPLOYED'].max()
print(f"Valor máximo em DAYS_EMPLOYED após o tratamento: {novo_max}")

Clientes identificados com o marcador de sistema (365243): 6135
Valor máximo em DAYS_EMPLOYED após o tratamento: 0


### Análise do Tratamento de Outliers

A execução do código confirmou a presença de **6.135 clientes** (aproximadamente 17% da nossa base de modelagem) registrados com o marcador de sistema `365243`. 

O sucesso do tratamento é evidenciado pela validação final, que retornou o valor máximo da coluna `DAYS_EMPLOYED` como **0**. Analiticamente, esse é o comportamento correto e esperado pela seguinte lógica:

* **Escala Temporal Coerente:** Como o banco de dados registra o tempo em dias contados retroativamente a partir da data de solicitação do crédito (valores negativos), o teto lógico para essa variável é exatamente `0`. Ele passa a representar fielmente os indivíduos que não possuem dias acumulados de emprego formal (desempregados ou aposentados).
* **Preservação do Algoritmo:** Ao substituir o salto artificial de 1.000 anos positivos por 0, garantimos que os algoritmos de *Machine Learning* interpretem a variável de forma linear e proporcional, sem que os cálculos de distância ou gradiente sejam severamente distorcidos por um *placeholder* do sistema.

## 3. Feature engineering

**Justificativa:** _preencher._

## 4. Normalização / padronização

**Escolha e justificativa:** _StandardScaler, MinMaxScaler ou nenhum?_

Aplique preferencialmente dentro de um `Pipeline` no notebook 03 — ajustar o scaler antes do split vaza informação do teste para o treino.

In [ ]:
# novas variáveis derivadas

## 4. Salvar dataset tratado

In [ ]:
PROCESSED.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED / "dataset_tratado.csv", index=False)